# Lab 1: Beyond Keywords

## This lab has 5 steps.

1. Prepare the full-text catalog and key.
2. Index the source text.
3. Compare matching words with a semantic gap.
4. Inspect the prepared identity boundary.
5. Register the external embedding model.

## Lab Goals

Discover where keyword matching leaves a semantic gap, then connect SQL to the model that will help us search by meaning.

## The challenge

A case describes **crushing chest pain**. Someone else describes the feeling as
**"an elephant is standing on my torso."** People can connect those descriptions,
but a word-based search may not. We will follow that same example through all
four notebooks, examining what improves and what still needs human judgment.

Full-text search is useful for matching vocabulary and will become one of
our hybrid retrieval signals. First, let's explore where it works well and
where semantic retrieval can complement it.

![Case text becomes chunks and vectors; a query follows lexical and vector retrieval paths that merge through RRF, with optional intent extraction before search.](../img/retrieval-overview.png)

The journey: prepare the data once, retrieve by words and meaning, then fuse the rankings. Intent extraction is an optional enhancement, not a prerequisite.

## Starting point

**Complete [00: SQL setup](00-setup.ipynb) first** on your pre-deployed VM.
Its SQL cells create the master key and save your Foundry resource endpoint
in a managed-identity credential; no attendee PowerShell is required.
You do not paste a Foundry URL into this notebook; Steps 4 and 5 read it
from that credential. The host supplies the resources and synthetic cases.

Use the MSSQL kernel in VS Code and connect to the fresh, prepared
`Healthcare` database. The lab VM is expected to be provisioned at login;
the readiness check below verifies that the required database state actually
exists. Do not run the labs against `master` or a shared customer database.

Run one SQL cell at a time, not **Run All**. Variables do not persist between
cells; the saved SQL objects carry state forward. The [lab guide](README.md)
covers setup, costs, and cleanup.

> The 207 cases are synthetic. This is a retrieval exercise, not medical
> advice, diagnosis, treatment guidance, or a clinical decision system.

**Learn more:** [SQL notebooks in VS Code](https://learn.microsoft.com/en-us/sql/tools/visual-studio-code-extensions/mssql/mssql-sql-notebooks?view=sql-server-ver17).


## Step 00: Verify this lab is ready to run

A fresh VM should already have the required database setup. Check that state before creating or calling anything.
This read-only checkpoint reports one ordered Result column and makes no
billed model calls. Every required check must say **Yes**.

A **No** or **Skipped** result produces controlled error 51000 after the
grid. Follow its diagnostic and rerun the checkpoint; the script does not
provision or repair resources for you.


In [ ]:
SET NOCOUNT ON;
DECLARE @Checks TABLE (
    Sort INT IDENTITY PRIMARY KEY, [Check] NVARCHAR(100),
    Passed BIT NOT NULL, Skipped BIT NOT NULL DEFAULT 0, Details NVARCHAR(2048)
);
DECLARE @DatabaseReady BIT = CASE WHEN DB_NAME() = N'Healthcare' THEN 1 ELSE 0 END;
DECLARE @Passed BIT;
INSERT INTO @Checks ([Check], Passed, Details)
VALUES (N'Workshop database', @DatabaseReady, N'Connect the MSSQL kernel to Healthcare, not master.');

SET @Passed = 0;
IF @DatabaseReady = 1 AND (OBJECT_ID(N'dbo.Cases', N'U') IS NOT NULL)
    EXEC sys.sp_executesql
        N'SELECT @Ok = CASE WHEN (SELECT COUNT(*) FROM dbo.Cases) = 207 THEN 1 ELSE 0 END;',
        N'@Ok BIT OUTPUT', @Ok = @Passed OUTPUT;
INSERT INTO @Checks ([Check], Passed, Skipped, Details)
VALUES (N'Seeded cases', @Passed,
    CASE WHEN @DatabaseReady = 0 OR NOT (OBJECT_ID(N'dbo.Cases', N'U') IS NOT NULL) THEN 1 ELSE 0 END,
    N'Ask the facilitator to prepare all 207 synthetic cases; verify the database in notebook 00.');

SET @Passed = 0;
IF @DatabaseReady = 1 AND (1 = 1)
    EXEC sys.sp_executesql
        N'SELECT @Ok = CASE WHEN (SELECT COUNT(*) FROM sys.database_scoped_credentials
    WHERE credential_identity = N''Managed Identity'' AND name LIKE N''https://foundry-%-cu.services.ai.azure.com'') = 1 THEN 1 ELSE 0 END;',
        N'@Ok BIT OUTPUT', @Ok = @Passed OUTPUT;
INSERT INTO @Checks ([Check], Passed, Skipped, Details)
VALUES (N'Prepared Foundry credential', @Passed,
    CASE WHEN @DatabaseReady = 0 OR NOT (1 = 1) THEN 1 ELSE 0 END,
    N'Run instructions/00-setup.ipynb Steps 2 and 3 to create the master key and Foundry endpoint credential in Healthcare.');

SELECT CONCAT([Check], N': ', CASE
    WHEN Skipped = 1 THEN CONCAT(N'Skipped - ', Details)
    WHEN Passed = 1 THEN N'Yes'
    ELSE CONCAT(N'No - ', Details)
END) AS Result
FROM @Checks ORDER BY Sort;
IF EXISTS (SELECT 1 FROM @Checks WHERE Passed = 0)
    THROW 51000, 'Resolve the failed or skipped required checks above.', 1;


### Expected Output
Estimated time: 1-5 seconds.

| Result |
|---|
| Workshop database: Yes |
| Seeded cases: Yes |
| Prepared Foundry credential: Yes |


## Step 1: Prepare the full-text catalog and key

Full-text indexing needs a catalog and a unique, non-null document key.
We will add a catalog and a key index owned by this track to the already
seeded `dbo.Cases` table. The statements leave all 207 case descriptions
unchanged and reuse these objects if they already exist.

This is still ordinary lexical retrieval: no model is called, and no text
has been chunked or embedded.


In [ ]:
IF NOT EXISTS (SELECT 1 FROM sys.fulltext_catalogs WHERE name = N'RetrievalCatalog')
    CREATE FULLTEXT CATALOG RetrievalCatalog;
IF NOT EXISTS (SELECT 1 FROM sys.indexes WHERE object_id = OBJECT_ID(N'dbo.Cases') AND name = N'UX_RetrievalCases_Id')
    CREATE UNIQUE INDEX UX_RetrievalCases_Id ON dbo.Cases(Id);


### Expected result
Estimated time: 1-5 seconds.

Both statements complete. Rerunning them does not recreate existing objects.


### Inspect the result

Before using the catalog and key, inspect their metadata. The two small
grids should show `RetrievalCatalog` and `UX_RetrievalCases_Id`, with
`is_unique = 1` for the index.

These checks establish the full-text prerequisites, not a populated
full-text index. We build that next.


In [ ]:
SELECT name FROM sys.fulltext_catalogs WHERE name = N'RetrievalCatalog';
SELECT name, is_unique FROM sys.indexes
WHERE object_id = OBJECT_ID(N'dbo.Cases') AND name = N'UX_RetrievalCases_Id';


### Expected result
Estimated time: 1-5 seconds.

`RetrievalCatalog` and `UX_RetrievalCases_Id` exist; `is_unique` is 1.


## Step 2: Index the source text

Index the **Complaint** and **Symptoms** columns on the source cases.
This lets us explore lexical retrieval before data preparation; the full-text
demonstration uses source text directly, without requiring chunks from a later notebook.

A table can have only one full-text index. If an incompatible index already
exists, this cell stops rather than replacing shared configuration.
Creation starts asynchronous population, so wait for the next check before
drawing conclusions from an empty result.

**Learn more:** [Query with full-text search](https://learn.microsoft.com/en-us/sql/relational-databases/search/query-with-full-text-search?view=sql-server-ver17).


In [ ]:
IF EXISTS (SELECT 1 FROM sys.fulltext_indexes
    WHERE object_id = OBJECT_ID(N'dbo.Cases')
      AND fulltext_catalog_id <> (SELECT fulltext_catalog_id FROM sys.fulltext_catalogs WHERE name = N'RetrievalCatalog'))
    THROW 51000, 'Cases already has another full-text catalog. Use a dedicated workshop database.', 1;
IF NOT EXISTS (SELECT 1 FROM sys.fulltext_indexes WHERE object_id = OBJECT_ID(N'dbo.Cases'))
    CREATE FULLTEXT INDEX ON dbo.Cases (Complaint LANGUAGE 1033, Symptoms LANGUAGE 1033)
        KEY INDEX UX_RetrievalCases_Id ON RetrievalCatalog;


### Expected result
Estimated time: 1-5 seconds.

The index is created. Population continues asynchronously.


### Inspect the result

Full-text index creation can finish before its population does. This
cell waits for that initial population, with a bounded timeout, then checks
that both source columns are indexed in English.

Inspect the two column names and `language_id = 1033`. If the timeout fires,
investigate population and rerun this check; do not interpret an unfinished
index as a search-quality failure.


In [ ]:
DECLARE @Deadline DATETIME2 = DATEADD(SECOND, 120, SYSDATETIME());
WHILE EXISTS (SELECT 1 FROM sys.fulltext_indexes WHERE object_id = OBJECT_ID(N'dbo.Cases')
    AND (has_crawl_completed = 0 OR OBJECTPROPERTYEX(object_id, 'TableFulltextPopulateStatus') <> 0))
BEGIN
    IF SYSDATETIME() >= @Deadline
        THROW 51000, 'Source full-text population timed out. Inspect population, then rerun this cell.', 1;
    WAITFOR DELAY '00:00:02';
END;
IF (SELECT COUNT(*) FROM sys.fulltext_index_columns
    WHERE object_id = OBJECT_ID(N'dbo.Cases')
      AND column_id IN (COLUMNPROPERTY(OBJECT_ID(N'dbo.Cases'), N'Complaint', 'ColumnId'),
                        COLUMNPROPERTY(OBJECT_ID(N'dbo.Cases'), N'Symptoms', 'ColumnId'))
      AND language_id = 1033) <> 2
    THROW 51000, 'Cases must index Complaint and Symptoms in English (1033).', 1;
SELECT c.name AS IndexedColumn, f.language_id
FROM sys.fulltext_index_columns f JOIN sys.columns c
    ON c.object_id = f.object_id AND c.column_id = f.column_id
WHERE f.object_id = OBJECT_ID(N'dbo.Cases');


### Expected result
Estimated time: 1-120 seconds.

Two indexed columns, each with `language_id = 1033`.


## Step 3: Compare matching words with a semantic gap

![A chest-pain keyword query matches words in source documents and returns case descriptions containing those terms.](../img/lexical-search.png)

Conceptual lexical-search flow: the highlighted words show vocabulary matches,
not measured rankings or the exact output of the queries below.

Let's compare vocabulary with an idiom. The first query uses words
present in case 1. The second describes a related sensation without those
words. Both use `FREETEXTTABLE`, which returns matching document keys and
lexical ranks for us to join back to the source text.

**Predict:** Which query will retrieve the anchor case? What might the word
"standing" match instead?

Compare the case IDs and descriptions, not only the numbers in the score
column. Empty results or matches to a different meaning reveal a retrieval limitation; they
do not establish that no relevant case exists.

**Learn more:** [Query with full-text search](https://learn.microsoft.com/en-us/sql/relational-databases/search/query-with-full-text-search?view=sql-server-ver17).


In [ ]:
SELECT TOP (5) c.Id, c.Complaint, f.[RANK] AS LexicalScore
FROM FREETEXTTABLE(dbo.Cases, (Complaint, Symptoms), N'crushing chest pain') f
JOIN dbo.Cases c ON c.Id = f.[KEY]
ORDER BY f.[RANK] DESC, c.Id;

SELECT TOP (5) c.Id, c.Complaint, f.[RANK] AS LexicalScore
FROM FREETEXTTABLE(dbo.Cases, (Complaint, Symptoms), N'An elephant is standing on my torso') f
JOIN dbo.Cases c ON c.Id = f.[KEY]
ORDER BY f.[RANK] DESC, c.Id;


### Expected result
Estimated time: 1-5 seconds.

The first query retrieves chest-pain cases. The second may be empty or favor literal matches such as standing. Exact ranks depend on the full-text engine; rehearse the contrast rather than expecting a fixed ranking.


## Step 4: Inspect the prepared identity boundary

To bridge the semantic gap, we need a model connection. Before
registering it, inspect the credential that the prepared environment supplies.
Its name is the actual Foundry resource endpoint, and its identity should be
`Managed Identity`; no token or API key is displayed.

![The participant authenticates to SQL with Microsoft Entra; the SQL server managed identity accesses the registered embedding endpoint in Foundry through a database-scoped credential.](../img/model-access.png)

Two identities, two jobs: your Entra account runs SQL; the SQL server managed identity calls Foundry. The external model is an endpoint association, not a model running inside SQL.

Look for exactly one workshop Foundry credential. Do not derive its endpoint
from the SQL server name or another resource's name.

**Learn more:** [CREATE DATABASE SCOPED CREDENTIAL](https://learn.microsoft.com/en-us/sql/t-sql/statements/create-database-scoped-credential-transact-sql?view=sql-server-ver17).


In [ ]:
SELECT name AS FoundryEndpoint, credential_identity
FROM sys.database_scoped_credentials
WHERE credential_identity = N'Managed Identity'
    AND name LIKE N'https://foundry-%-cu.services.ai.azure.com';


### Expected result
Estimated time: 1-5 seconds.

Exactly one workshop Foundry credential using Managed Identity. No secrets are displayed.


## Step 5: Register the external embedding model

Register `embedding_openai_text3_small` using the existing credential
and its authoritative endpoint. Later notebooks can invoke this model name
without asking you to re-enter the endpoint.

Registration describes where inference will happen; it does not itself call
the model. The first real smoke request is in notebook 2. Matching existing
configuration is reused, while conflicting configuration stops explicitly
rather than being dropped.

**Learn more:** [CREATE EXTERNAL MODEL](https://learn.microsoft.com/en-us/sql/t-sql/statements/create-external-model-transact-sql?view=sql-server-ver17).


In [ ]:
DECLARE @Endpoint SYSNAME, @Count INT;
SELECT @Endpoint = MAX(name), @Count = COUNT(*)
FROM sys.database_scoped_credentials
WHERE credential_identity = N'Managed Identity'
    AND name LIKE N'https://foundry-%-cu.services.ai.azure.com';
IF @Count <> 1
    THROW 51000, 'Complete instructions/00-setup.ipynb; exactly one workshop Foundry managed-identity credential is required.', 1;
DECLARE @Location NVARCHAR(4000) = @Endpoint
    + N'/openai/deployments/text-embedding-3-small/embeddings?api-version=2024-02-01';
IF EXISTS (SELECT 1 FROM sys.external_models m
    LEFT JOIN sys.database_scoped_credentials c ON c.credential_id = m.credential_id
    WHERE m.name = N'embedding_openai_text3_small'
      AND (m.location <> @Location OR c.name IS NULL OR c.name <> @Endpoint))
    THROW 51000, 'Existing embedding model uses different configuration. Review it; do not replace shared configuration.', 1;
IF NOT EXISTS (SELECT 1 FROM sys.external_models WHERE name = N'embedding_openai_text3_small')
BEGIN
    DECLARE @SQL NVARCHAR(MAX) = N'CREATE EXTERNAL MODEL embedding_openai_text3_small
WITH (LOCATION = ''' + REPLACE(@Location, '''', '''''') + N''',
    API_FORMAT = ''Azure OpenAI'', MODEL_TYPE = EMBEDDINGS,
    MODEL = ''text-embedding-3-small'', CREDENTIAL = ' + QUOTENAME(@Endpoint) + N');';
    EXEC sys.sp_executesql @SQL;
END;


### Expected result
Estimated time: 1-5 seconds.

The external model is registered, or a matching existing association is reused. This cell makes no inference request.


### Inspect the result

Inspect the model and credential together. A model name by itself is
not enough: the deployment URL must be associated with the expected
managed-identity credential.

Expect one row showing `embedding_openai_text3_small`, its embeddings URL,
and `Managed Identity`. This is a configuration check, not proof of
connectivity; notebook 2 will make the first explicit inference call.


In [ ]:
SELECT m.name AS ExternalModel, m.location, c.name AS Credential, c.credential_identity
FROM sys.external_models m
JOIN sys.database_scoped_credentials c ON c.credential_id = m.credential_id
WHERE m.name = N'embedding_openai_text3_small';


### Expected result
Estimated time: 1-5 seconds.

One correctly associated model row. Metadata is not a connectivity test: the first billed smoke call happens explicitly in Lab 02.


## Final Step: Validate this lab is complete

Before moving on, verify the persistent handoffs this notebook owns or depends on.
This read-only checkpoint reports one ordered Result column and makes no
billed model calls. Every required check must say **Yes**.

A **No** or **Skipped** result produces controlled error 51000 after the
grid. Follow its diagnostic and rerun the checkpoint; the script does not
provision or repair resources for you.

This checks persisted state, not live connectivity, search relevance, or index use. Those require inspecting the explicit exercises above.


In [ ]:
SET NOCOUNT ON;
DECLARE @Checks TABLE (
    Sort INT IDENTITY PRIMARY KEY, [Check] NVARCHAR(100),
    Passed BIT NOT NULL, Skipped BIT NOT NULL DEFAULT 0, Details NVARCHAR(2048)
);
DECLARE @DatabaseReady BIT = CASE WHEN DB_NAME() = N'Healthcare' THEN 1 ELSE 0 END;
DECLARE @Passed BIT;
INSERT INTO @Checks ([Check], Passed, Details)
VALUES (N'Workshop database', @DatabaseReady, N'Connect the MSSQL kernel to Healthcare, not master.');

SET @Passed = 0;
IF @DatabaseReady = 1 AND (OBJECT_ID(N'dbo.Cases', N'U') IS NOT NULL)
    EXEC sys.sp_executesql
        N'SELECT @Ok = CASE WHEN (SELECT COUNT(*) FROM dbo.Cases) = 207 THEN 1 ELSE 0 END;',
        N'@Ok BIT OUTPUT', @Ok = @Passed OUTPUT;
INSERT INTO @Checks ([Check], Passed, Skipped, Details)
VALUES (N'Seeded cases', @Passed,
    CASE WHEN @DatabaseReady = 0 OR NOT (OBJECT_ID(N'dbo.Cases', N'U') IS NOT NULL) THEN 1 ELSE 0 END,
    N'Ask the facilitator to prepare all 207 synthetic cases; verify the database in notebook 00.');

SET @Passed = 0;
IF @DatabaseReady = 1 AND (OBJECT_ID(N'dbo.Cases', N'U') IS NOT NULL)
    EXEC sys.sp_executesql
        N'SELECT @Ok = CASE WHEN EXISTS (SELECT 1 FROM sys.fulltext_indexes
    WHERE object_id = OBJECT_ID(N''dbo.Cases'') AND has_crawl_completed = 1)
    AND OBJECTPROPERTYEX(OBJECT_ID(N''dbo.Cases''), ''TableFulltextPopulateStatus'') = 0 THEN 1 ELSE 0 END;',
        N'@Ok BIT OUTPUT', @Ok = @Passed OUTPUT;
INSERT INTO @Checks ([Check], Passed, Skipped, Details)
VALUES (N'Source full-text index populated', @Passed,
    CASE WHEN @DatabaseReady = 0 OR NOT (OBJECT_ID(N'dbo.Cases', N'U') IS NOT NULL) THEN 1 ELSE 0 END,
    N'Complete retrieval Lab 01; wait for source full-text population.');

SET @Passed = 0;
IF @DatabaseReady = 1 AND (1 = 1)
    EXEC sys.sp_executesql
        N'SELECT @Ok = CASE WHEN EXISTS (SELECT 1 FROM sys.external_models m
    JOIN sys.database_scoped_credentials c ON c.credential_id = m.credential_id
    WHERE m.name = N''embedding_openai_text3_small''
      AND c.credential_identity = N''Managed Identity'') THEN 1 ELSE 0 END;',
        N'@Ok BIT OUTPUT', @Ok = @Passed OUTPUT;
INSERT INTO @Checks ([Check], Passed, Skipped, Details)
VALUES (N'External embedding model', @Passed,
    CASE WHEN @DatabaseReady = 0 OR NOT (1 = 1) THEN 1 ELSE 0 END,
    N'Complete retrieval Lab 01 and inspect the model/credential association.');

SELECT CONCAT([Check], N': ', CASE
    WHEN Skipped = 1 THEN CONCAT(N'Skipped - ', Details)
    WHEN Passed = 1 THEN N'Yes'
    ELSE CONCAT(N'No - ', Details)
END) AS Result
FROM @Checks ORDER BY Sort;
IF EXISTS (SELECT 1 FROM @Checks WHERE Passed = 0)
    THROW 51000, 'Resolve the failed or skipped required checks above.', 1;


### Expected Output
Estimated time: 1-5 seconds.

| Result |
|---|
| Workshop database: Yes |
| Seeded cases: Yes |
| Source full-text index populated: Yes |
| External embedding model: Yes |


[Session guide](README.md) | Next: [Notebook 2](02-case-notes-to-vectors.ipynb)
